# LeetCode 965 — Univalued Binary Tree bằng Python

**Link đề:** https://leetcode.com/problems/univalued-binary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isUnivalTree(self, root: TreeNode | None) -> bool:
        ...
```

## Mục tiêu học
1. Làm quen với **cây nhị phân** (binary tree): node, `val`, `left`, `right`, `None`; biết dựng cây từ dạng list của LeetCode.
2. Viết **đệ quy trên cây** theo khuôn "trường hợp cơ sở + gọi cho con trái/phải + gộp kết quả".
3. Duyệt cây theo 3 cách: DFS đệ quy, DFS bằng **stack**, BFS bằng **queue** (`collections.deque`).
4. Thấy được sức mạnh của **dừng sớm** (short-circuit `and`) khi chỉ cần một phản ví dụ.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Một cây nhị phân là **uni-valued** (đơn trị) nếu **mọi** node trong cây có **cùng** giá trị. Cho `root`, trả về `True` nếu cây đơn trị, ngược lại `False`.

Ví dụ 1: `root = [1,1,1,1,1,null,1]` → `True`
```text
        1
       / \
      1   1
     / \   \
    1   1   1
```

Ví dụ 2: `root = [2,2,2,5,2]` → `False` (có node giá trị 5)
```text
        2
       / \
      2   2
     / \
    5   2
```

**Constraints:** số node trong `[1, 100]`; `0 <= Node.val < 100`.

## 2. Nền tảng: cây nhị phân trong Python

### 2.1. Lớp `TreeNode`

Mỗi node là một **object** có 3 thuộc tính: `val` (giá trị), `left` (con trái), `right` (con phải). Không có con thì là `None`. LeetCode đã định nghĩa sẵn lớp này; để notebook chạy độc lập ta tự viết lại.

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng tay cây:   7
#                / \
#               7   7
root = TreeNode(7, TreeNode(7), TreeNode(7))
print(root.val, root.left.val, root.right.val)
print("root.left.left =", root.left.left)

7 7 7
root.left.left = None


### 2.2. Dựng cây từ list kiểu LeetCode

LeetCode ghi cây theo **thứ tự từng tầng** (level order), `None` (trong đề là `null`) đánh dấu vị trí trống. Ta dựng bằng một **hàng đợi** (queue): lấy lần lượt từng node ra và gán cho nó hai con tiếp theo trong list.

`collections.deque` là hàng đợi hai đầu: `append(x)` thêm vào cuối, `popleft()` lấy ra ở đầu — cả hai O(1).

In [2]:
from collections import deque

q = deque()
q.append("A"); q.append("B"); q.append("C")
print("ban đầu:", list(q))
front = q.popleft()
print("popleft lấy ra:", front, "| còn lại:", list(q))

ban đầu: ['A', 'B', 'C']
popleft lấy ra: A | còn lại: ['B', 'C']


In [3]:
def build_tree(values: list):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_list(root) -> list:
    # Chiều ngược lại: cây -> list level order (bỏ None thừa ở cuối)
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

t1 = build_tree([1, 1, 1, 1, 1, None, 1])
t2 = build_tree([2, 2, 2, 5, 2])
print(to_list(t1))
print(to_list(t2))

[1, 1, 1, 1, 1, None, 1]
[2, 2, 2, 5, 2]


### 2.3. In cây để nhìn cho rõ

Hàm `show` in cây xoay ngang 90°: con phải ở trên, con trái ở dưới, mỗi tầng thụt vào 4 ký tự. Đây là **đệ quy** đầu tiên của ta: in cây con phải, in node, rồi in cây con trái.

In [4]:
def show(node, depth=0):
    if node is None:
        return
    show(node.right, depth + 1)
    print("    " * depth + str(node.val))
    show(node.left, depth + 1)

show(t2)

    2
2
        2
    2
        5


### 2.4. Khuôn đệ quy trên cây

Gần như mọi hàm đệ quy trên cây nhị phân có dạng:

```python
def f(node):
    if node is None:          # 1. trường hợp cơ sở: cây rỗng
        return <giá trị cho cây rỗng>
    a = f(node.left)          # 2. giải bài toán cho cây con trái
    c = f(node.right)         #    và cây con phải
    return <gộp node.val, a, c>   # 3. gộp
```

Ví dụ đếm số node: cây rỗng có 0 node; ngược lại `1 + đếm(trái) + đếm(phải)`.

In [5]:
def count_nodes(node) -> int:
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

print("t1 có", count_nodes(t1), "node; t2 có", count_nodes(t2), "node")

t1 có 6 node; t2 có 5 node


## 3. Cách trực giác (baseline) — gom hết giá trị rồi đếm số loại

Ý tưởng tự nhiên: duyệt toàn bộ cây, bỏ mọi giá trị vào một **set**. Cây đơn trị ⇔ set có đúng **1** phần tử.

In [6]:
def collect(node, values: set) -> None:
    if node is None:
        return
    values.add(node.val)
    collect(node.left, values)
    collect(node.right, values)

def is_unival_set(root) -> bool:
    values = set()
    collect(root, values)
    return len(values) == 1

print(is_unival_set(t1), is_unival_set(t2))

True False


Độ phức tạp: thăm mỗi node 1 lần → O(n) thời gian; set có thể chứa đến O(n) giá trị và đệ quy sâu O(h) → O(n) bộ nhớ trường hợp xấu.

## 4. Nút thắt

Baseline có hai điểm lãng phí:

1. **Không dừng sớm:** ngay khi thấy một giá trị khác, ta đã biết đáp án là `False`, nhưng baseline vẫn duyệt hết cây.
2. **Lưu thừa:** ta không cần biết *tất cả* giá trị, chỉ cần so mỗi node với **một** giá trị chuẩn.

## 5. Insight

Cây đơn trị ⇔ **mọi node có giá trị bằng `root.val`**.

Hoặc phát biểu cục bộ (đẹp hơn cho đệ quy): cây đơn trị ⇔ **mỗi cạnh cha–con nối hai node bằng giá trị nhau**. Vì cây liên thông, "mọi cạnh bằng nhau" kéo theo "mọi node bằng root" (bắc cầu dọc đường đi từ root xuống).

Kết hợp với toán tử `and` **ngắn mạch** (short-circuit): `A and B` không tính `B` nếu `A` đã `False` → dừng sớm miễn phí.

In [7]:
def noisy(name, value):
    print("   tính", name)
    return value

print("Kết quả:", noisy("A", False) and noisy("B", True))
print("Kết quả:", noisy("A", True) and noisy("B", True))

   tính A
Kết quả: False
   tính A
   tính B
Kết quả: True


## 6. Lời giải chính — xây từng đoạn rồi ghép

Viết hàm phụ `dfs(node, target)`: "cây con gốc `node` có mọi giá trị bằng `target` không?"

**Đoạn 1 — cơ sở:** cây rỗng không chứa giá trị sai nào → `True`.
```python
if node is None:
    return True
```
**Đoạn 2 — node hiện tại:** nếu `node.val != target` → `False` ngay.

**Đoạn 3 — gộp:** cây con trái **và** cây con phải đều phải đúng.
```python
return dfs(node.left, target) and dfs(node.right, target)
```

In [8]:
def is_unival(root) -> bool:
    target = root.val

    def dfs(node) -> bool:
        if node is None:                       # Đoạn 1
            return True
        if node.val != target:                 # Đoạn 2
            return False
        return dfs(node.left) and dfs(node.right)   # Đoạn 3

    return dfs(root)

print(is_unival(t1), is_unival(t2))

True False


Hàm `dfs` được định nghĩa **bên trong** `is_unival` (hàm lồng / nested function) nên đọc được biến `target` của hàm ngoài mà không cần truyền tham số — gọi là **closure**.

## 7. Trace từng bước

Cây Ví dụ 2 `[2,2,2,5,2]`, `target = 2`. Thứ tự gọi (DFS: node → trái → phải):

| Bước | Node | val | So với 2 | Hành động |
|:---:|---|:---:|:---:|---|
| 1 | root | 2 | = | gọi con trái |
| 2 | root.left | 2 | = | gọi con trái |
| 3 | root.left.left | 5 | ≠ | **trả False** |
| — | | | | `and` ngắn mạch: không gọi root.left.right, không gọi root.right |

In [9]:
def trace_unival(root) -> bool:
    target = root.val
    print("target =", target)

    def dfs(node, path, depth):
        pad = "  " * depth
        if node is None:
            print(f"{pad}{path}: None -> True")
            return True
        if node.val != target:
            print(f"{pad}{path}: val={node.val} != {target} -> False (dừng)")
            return False
        print(f"{pad}{path}: val={node.val} == {target} -> xét con")
        return dfs(node.left, path + ".L", depth + 1) and dfs(node.right, path + ".R", depth + 1)

    result = dfs(root, "root", 0)
    print("Kết quả:", result)
    return result

trace_unival(t2)
print()
trace_unival(build_tree([1, 1, 1]))

target = 2
root: val=2 == 2 -> xét con
  root.L: val=2 == 2 -> xét con
    root.L.L: val=5 != 2 -> False (dừng)
Kết quả: False

target = 1
root: val=1 == 1 -> xét con
  root.L: val=1 == 1 -> xét con
    root.L.L: None -> True
    root.L.R: None -> True
  root.R: val=1 == 1 -> xét con
    root.R.L: None -> True
    root.R.R: None -> True
Kết quả: True


True

Ở cây thứ nhất chỉ có 3 node được thăm (trên tổng 5). Ở cây thứ hai mọi node đều được thăm, kể cả các vị trí `None` — đó là các lần chạm trường hợp cơ sở.

## 8. Code nộp LeetCode

LeetCode tự cung cấp `TreeNode`; khi nộp chỉ dán phần `class Solution`.

### Cách A — DFS đệ quy so với `root.val`

In [10]:
class Solution:
    def isUnivalTree(self, root: TreeNode | None) -> bool:
        target = root.val

        def dfs(node):
            if node is None:
                return True
            if node.val != target:
                return False
            return dfs(node.left) and dfs(node.right)

        return dfs(root)

print(Solution().isUnivalTree(build_tree([1, 1, 1, 1, 1, None, 1])))
print(Solution().isUnivalTree(build_tree([2, 2, 2, 5, 2])))

True
False


### Cách B — đệ quy so cha với con (không cần biến `target`)

Dùng phát biểu "mọi cạnh cha–con bằng nhau": node hợp lệ nếu con trái (nếu có) cùng giá trị, con phải (nếu có) cùng giá trị, và hai cây con cũng hợp lệ.

In [11]:
class SolutionEdge:
    def isUnivalTree(self, root: TreeNode | None) -> bool:
        if root is None:
            return True
        if root.left and root.left.val != root.val:
            return False
        if root.right and root.right.val != root.val:
            return False
        return self.isUnivalTree(root.left) and self.isUnivalTree(root.right)

print(SolutionEdge().isUnivalTree(t1), SolutionEdge().isUnivalTree(t2))

True False


### Cách C — DFS bằng stack (không đệ quy)

**Stack** (ngăn xếp) là "vào sau ra trước": list Python dùng `append` để đẩy vào, `pop()` để lấy phần tử cuối. Thay vì để Python quản lý lời gọi đệ quy, ta tự giữ danh sách node chờ xét.

In [12]:
st = [1, 2]
st.append(3)
print("pop ->", st.pop(), "| còn", st)

pop -> 3 | còn [1, 2]


In [13]:
class SolutionStack:
    def isUnivalTree(self, root: TreeNode | None) -> bool:
        stack = [root]
        while stack:
            node = stack.pop()
            if node.val != root.val:
                return False
            if node.left:
                stack.append(node.left)
            if node.right:
                stack.append(node.right)
        return True

print(SolutionStack().isUnivalTree(t1), SolutionStack().isUnivalTree(t2))

True False


### Cách D — BFS theo tầng với `deque`

Giống Cách C nhưng dùng hàng đợi: xét node theo từng tầng từ trên xuống. Hữu ích khi phản ví dụ thường nằm gần gốc.

In [14]:
class SolutionBFS:
    def isUnivalTree(self, root: TreeNode | None) -> bool:
        queue = deque([root])
        while queue:
            node = queue.popleft()
            if node.val != root.val:
                return False
            for child in (node.left, node.right):
                if child:
                    queue.append(child)
        return True

print(SolutionBFS().isUnivalTree(t1), SolutionBFS().isUnivalTree(t2))

True False


## 9. Vì sao đúng?

**Cách A.** Mệnh đề: `dfs(node)` trả `True` ⇔ mọi node trong cây con gốc `node` có giá trị `target`. Chứng minh quy nạp theo chiều cao:
- Cây rỗng: không có node nào sai → `True` đúng.
- Cây khác rỗng: đúng ⇔ `node.val == target` **và** hai cây con đều đúng — chính xác là điều code kiểm tra.

Áp dụng cho `root` với `target = root.val` → đúng định nghĩa cây đơn trị.

**Cách B.** Nếu mọi cạnh cha–con bằng nhau, đi từ root đến node bất kỳ qua một đường các cạnh bằng nhau → node đó bằng root. Ngược lại, nếu có cạnh khác nhau thì hai đầu cạnh khác giá trị → không đơn trị.

**Cách C/D.** Invariant: mọi node đã lấy ra khỏi stack/queue đều bằng `root.val`; mọi node chưa xét đều nằm trong stack/queue hoặc là con cháu của node trong đó. Vòng lặp kết thúc khi stack rỗng → đã xét mọi node.

## 10. Độ phức tạp

Gọi `n` = số node, `h` = chiều cao cây (`h` từ `log n` khi cân bằng đến `n` khi cây lệch thành "dây").

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Baseline set | O(n) | O(n) | không dừng sớm |
| A. DFS so root | O(n) | O(h) call stack | dừng sớm |
| B. Cha–con | O(n) | O(h) | không cần biến ngoài |
| C. Stack | O(n) | O(h) đến O(n) | không lo giới hạn đệ quy |
| D. BFS | O(n) | O(w) — `w` = độ rộng tầng lớn nhất | tìm phản ví dụ gần gốc nhanh |

Với `n ≤ 100`, giới hạn đệ quy mặc định của Python (~1000) là dư.

## 11. Bộ test

Nhóm test: ví dụ đề; một node; cây lệch trái/phải (dạng dây); giá trị khác ở lá sâu nhất; giá trị 0; khác giá trị ngay ở con của root. Cuối cùng đối chiếu ngẫu nhiên.

In [15]:
tests = [
    ([1, 1, 1, 1, 1, None, 1], True),
    ([2, 2, 2, 5, 2], False),
    ([7], True),
    ([0, 0, None, 0, None, 0], True),          # dây lệch trái
    ([3, None, 3, None, 3, None, 4], False),   # khác ở lá sâu nhất bên phải
    ([9, 8], False),
    ([9, None, 9], True),
    ([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2], False),
]
impls = {
    "set": is_unival_set, "chính": is_unival,
    "LC-A": Solution().isUnivalTree, "LC-B": SolutionEdge().isUnivalTree,
    "LC-C": SolutionStack().isUnivalTree, "LC-D": SolutionBFS().isUnivalTree,
}
for values, expected in tests:
    root = build_tree(values)
    for name, f in impls.items():
        got = f(root)
        assert got == expected, (name, values, got, expected)
    print(f"{str(values):50} -> {expected} PASS")
print("Tất cả test PASS!")

[1, 1, 1, 1, 1, None, 1]                           -> True PASS
[2, 2, 2, 5, 2]                                    -> False PASS
[7]                                                -> True PASS
[0, 0, None, 0, None, 0]                           -> True PASS
[3, None, 3, None, 3, None, 4]                     -> False PASS
[9, 8]                                             -> False PASS
[9, None, 9]                                       -> True PASS
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]      -> False PASS
Tất cả test PASS!


In [16]:
import random

def random_tree(rng, n, vals):
    # Sinh cây n node bằng cách gắn ngẫu nhiên vào chỗ trống
    root = TreeNode(rng.choice(vals))
    nodes = [root]
    while len(nodes) < n:
        parent = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(parent, side) is None:
            child = TreeNode(rng.choice(vals))
            setattr(parent, side, child)
            nodes.append(child)
    return root

rng = random.Random(965)
for _ in range(2000):
    root = random_tree(rng, rng.randint(1, 15), [4] * 12 + [5])
    results = {name: f(root) for name, f in impls.items()}
    assert len(set(results.values())) == 1, results
print("2000 cây ngẫu nhiên: mọi cách cho cùng kết quả")

2000 cây ngẫu nhiên: mọi cách cho cùng kết quả


`getattr(obj, "left")` đọc thuộc tính theo **tên dạng chuỗi**, `setattr(obj, "left", x)` gán thuộc tính — tiện khi chọn trái/phải ngẫu nhiên.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra con trực tiếp của root

In [17]:
def wrong_only_children(root):
    ok = True
    if root.left:
        ok = ok and root.left.val == root.val
    if root.right:
        ok = ok and root.right.val == root.val
    return ok

t = build_tree([2, 2, 2, 5, 2])
print("Sai :", wrong_only_children(t))
print("Đúng:", is_unival(t))

Sai : True
Đúng: False


**Vì sao sai:** node `5` nằm ở tầng 2, không phải con trực tiếp của root. **Sửa:** phải **đệ quy** xuống mọi tầng.

### Lỗi 2: Quên trường hợp `None` → `AttributeError`

In [18]:
def wrong_no_base(node, target):
    if node.val != target:          # node có thể là None!
        return False
    return wrong_no_base(node.left, target) and wrong_no_base(node.right, target)

try:
    wrong_no_base(build_tree([1, 1]), 1)
except AttributeError as e:
    print("Lỗi:", e)

Lỗi: 'NoneType' object has no attribute 'val'


**Sửa:** luôn viết trường hợp cơ sở `if node is None: return True` **trước** khi đọc `node.val`.

### Lỗi 3: Gọi đệ quy nhưng quên `return`/quên gộp kết quả

In [19]:
def wrong_forget_return(node, target):
    if node is None:
        return True
    if node.val != target:
        return False
    wrong_forget_return(node.left, target)     # kết quả bị bỏ đi!
    wrong_forget_return(node.right, target)
    return True

print("Sai :", wrong_forget_return(build_tree([2, 2, 2, 5, 2]), 2))

Sai : True


**Vì sao sai:** giá trị `False` của cây con bị vứt bỏ. **Sửa:** `return dfs(left) and dfs(right)`.

### Lỗi 4: So `root.left.val` mà không kiểm tra `root.left` tồn tại

Trong Cách B, phải viết `if root.left and root.left.val != root.val` — nhờ ngắn mạch `and`, `root.left.val` chỉ được đọc khi `root.left` không phải `None`.

## 13. Mẹo nhận dạng pattern

- Đề hỏi "**mọi** node có tính chất X không?" → đệ quy `X(node) and f(left) and f(right)`, cơ sở `None → True`.
- Đề hỏi "**có tồn tại** node có tính chất X không?" → `X(node) or f(left) or f(right)`, cơ sở `None → False`.
- Tính chất "toàn cục" (mọi node bằng nhau) thường viết lại được thành tính chất **cục bộ trên cạnh** (cha = con).
- Sợ đệ quy quá sâu (cây 10⁵ node dạng dây) → chuyển sang stack/queue.

## 14. Biến thể

1. **LeetCode 100 — Same Tree:** hai cây giống hệt? Đệ quy song song hai node.
2. **LeetCode 250 — Count Univalue Subtrees (Premium):** đếm số **cây con** đơn trị — đệ quy hậu thứ tự (post-order), mỗi node trả về "cây con của tôi có đơn trị không".
3. **LeetCode 687 — Longest Univalue Path:** đường đi dài nhất mà mọi node cùng giá trị.
4. Kiểm tra mọi node nằm trong khoảng `[lo, hi]`, hoặc mọi node chẵn — cùng khuôn "mọi node".

In [20]:
def count_unival_subtrees(root) -> int:
    count = 0
    def post(node):
        nonlocal count
        if node is None:
            return True
        left_ok, right_ok = post(node.left), post(node.right)   # gọi CẢ HAI, không ngắn mạch
        if not (left_ok and right_ok):
            return False
        if node.left and node.left.val != node.val:
            return False
        if node.right and node.right.val != node.val:
            return False
        count += 1
        return True
    post(root)
    return count

print("Số cây con đơn trị của [5,1,5,5,5,None,5]:", count_unival_subtrees(build_tree([5, 1, 5, 5, 5, None, 5])))

Số cây con đơn trị của [5,1,5,5,5,None,5]: 4


`nonlocal count` cho phép hàm lồng **gán lại** biến `count` của hàm ngoài (nếu chỉ đọc thì không cần).

## 15. Bài tập tự luyện (TODO)

### Bài 1 — Tổng giá trị
Viết `tree_sum(root)` trả về tổng mọi giá trị trong cây (cây rỗng → 0).

In [21]:
def tree_sum(root) -> int:
    # TODO
    pass

# assert tree_sum(build_tree([1, 2, 3, None, 4])) == 10
# assert tree_sum(None) == 0

### Bài 2 — Có tồn tại giá trị
Viết `contains(root, x)` trả `True` nếu có node mang giá trị `x`.

In [22]:
def contains(root, x: int) -> bool:
    # TODO
    pass

# assert contains(build_tree([2, 2, 2, 5, 2]), 5) == True
# assert contains(build_tree([2, 2, 2, 5, 2]), 7) == False

### Bài 3 — Mọi node chẵn
Viết `all_even(root)` dùng **stack**, không đệ quy.

In [23]:
def all_even(root) -> bool:
    # TODO
    pass

# assert all_even(build_tree([2, 4, 6])) == True
# assert all_even(build_tree([2, 4, 7])) == False

### Bài 4 — Đơn trị theo tầng
Viết `level_unival(root)` trả `True` nếu **trong mỗi tầng** mọi node cùng giá trị (các tầng có thể khác nhau). Dùng BFS.

In [24]:
def level_unival(root) -> bool:
    # TODO
    pass

# assert level_unival(build_tree([1, 2, 2, 3, 3, 3])) == True
# assert level_unival(build_tree([1, 2, 3])) == False

### Bài 5 — Same Tree
Viết `same_tree(p, q)` kiểm tra hai cây giống hệt (cấu trúc và giá trị).

In [25]:
def same_tree(p, q) -> bool:
    # TODO
    pass

# assert same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3])) == True
# assert same_tree(build_tree([1, 2]), build_tree([1, None, 2])) == False

### Bài 6 (nâng cao) — Đếm cây con đơn trị
Viết lại `count_unival(root)` (LeetCode 250) mà **không** nhìn mục 14.

In [26]:
def count_unival(root) -> int:
    # TODO
    pass

# assert count_unival(build_tree([5, 1, 5, 5, 5, None, 5])) == 4
# assert count_unival(build_tree([5, 5, 5, 5, 5, None, 5])) == 6

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | khuôn đệ quy | cây rỗng → 0 | `node.val + f(left) + f(right)` |
| 2 | "tồn tại" → `or` | cơ sở `None → False` | `node.val == x or f(left) or f(right)` |
| 3 | stack = list | đẩy con khác `None` | gặp số lẻ → `False` |
| 4 | xử lý từng tầng | `for _ in range(len(queue))` | gom giá trị tầng vào set, kiểm `len == 1` |
| 5 | đệ quy song song | cả hai `None` → True; một `None` → False | so `val` rồi `and` hai nhánh |
| 6 | post-order | phải gọi **cả hai** con trước khi gộp | đếm khi cả hai con OK và bằng giá trị node |

## 17. Đáp án tham khảo

In [27]:
def ans_tree_sum(root):
    if root is None:
        return 0
    return root.val + ans_tree_sum(root.left) + ans_tree_sum(root.right)

def ans_contains(root, x):
    if root is None:
        return False
    return root.val == x or ans_contains(root.left, x) or ans_contains(root.right, x)

def ans_all_even(root):
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        if node.val % 2:
            return False
        for child in (node.left, node.right):
            if child:
                stack.append(child)
    return True

def ans_level_unival(root):
    queue = deque([root] if root else [])
    while queue:
        level = set()
        for _ in range(len(queue)):
            node = queue.popleft()
            level.add(node.val)
            for child in (node.left, node.right):
                if child:
                    queue.append(child)
        if len(level) > 1:
            return False
    return True

def ans_same_tree(p, q):
    if p is None and q is None:
        return True
    if p is None or q is None:
        return False
    return p.val == q.val and ans_same_tree(p.left, q.left) and ans_same_tree(p.right, q.right)

def ans_count_unival(root):
    count = 0
    def post(node):
        nonlocal count
        if node is None:
            return True
        l, r = post(node.left), post(node.right)
        if l and r and (node.left is None or node.left.val == node.val) and (node.right is None or node.right.val == node.val):
            count += 1
            return True
        return False
    post(root)
    return count

assert ans_tree_sum(build_tree([1, 2, 3, None, 4])) == 10
assert ans_tree_sum(None) == 0
assert ans_contains(build_tree([2, 2, 2, 5, 2]), 5) == True
assert ans_contains(build_tree([2, 2, 2, 5, 2]), 7) == False
assert ans_all_even(build_tree([2, 4, 6])) == True
assert ans_all_even(build_tree([2, 4, 7])) == False
assert ans_level_unival(build_tree([1, 2, 2, 3, 3, 3])) == True
assert ans_level_unival(build_tree([1, 2, 3])) == False
assert ans_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3])) == True
assert ans_same_tree(build_tree([1, 2]), build_tree([1, None, 2])) == False
assert ans_count_unival(build_tree([5, 1, 5, 5, 5, None, 5])) == 4
assert ans_count_unival(build_tree([5, 5, 5, 5, 5, None, 5])) == 6
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isUnivalTree(self, root: TreeNode | None) -> bool:
        target = root.val
        def dfs(node):
            if node is None:
                return True
            return node.val == target and dfs(node.left) and dfs(node.right)
        return dfs(root)
```

Khuôn "mọi node thoả X": `None → True`; `X(node) and dfs(left) and dfs(right)`.

**Checklist trước khi submit**
- [ ] Có trường hợp cơ sở `None` trước khi đọc `.val`?
- [ ] Kết quả đệ quy được `return`/gộp, không bị bỏ đi?
- [ ] Kiểm tra **mọi tầng**, không chỉ con trực tiếp?
- [ ] Dùng `and` để dừng sớm khi gặp giá trị khác?
- [ ] Chỉ dán `class Solution` (LeetCode đã có `TreeNode`)?